In [1]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

# add more if needed

In [2]:
adata = ad.read_h5ad("/Users/dm954/Documents/code/mixed_diffusion/data/CITEseq/citeseq_preprocessed.h5ad")

In [ ]:
C_raw = adata.X   
meta = adata.obs  

In [4]:
C_raw.shape

(10000, 3000)

In [5]:
#### Start from here ####

C = C_raw / np.sqrt(C_raw.shape[0])
C = C.astype('float64')

In [6]:
# Perform a train-test split (100 random samples left out for testing)
from sklearn.model_selection import train_test_split

matrix = C

# Use the unnormalized matrix this time
labels_df = pd.DataFrame({'x': adata.obs['cell_labels']})

print(f"Original C_reduced shape: {matrix.shape}")
print(f"Original labels shape: {labels_df.shape}")

# Set random seed for reproducibility
np.random.seed(42)

# Get the number of samples (rows in C_reduced correspond to samples)
n_samples = matrix.shape[0]
print(f"Total number of samples: {n_samples}")

# Create indices for all samples
sample_indices = np.arange(n_samples)

# Split indices into train and test (1000 samples for test)
test_size = 1000
train_indices, test_indices = train_test_split(
    sample_indices, 
    test_size=test_size, 
    random_state=42,
    stratify=labels_df['x'].values  # Stratify by cell type to maintain class balance
)

print(f"Training samples: {len(train_indices)}")
print(f"Test samples: {len(test_indices)}")

# Split the data using the indices
C_train = matrix[train_indices]
C_test = matrix[test_indices]

# Split the labels correspondingly
labels_train = labels_df.iloc[train_indices]
labels_test = labels_df.iloc[test_indices]

# Print shapes and class distribution
print(f"\nTrain data shape: {C_train.shape}")
print(f"Test data shape: {C_test.shape}")
print(f"Train labels shape: {labels_train.shape}")
print(f"Test labels shape: {labels_test.shape}")

# Show class distribution in train and test sets
print(f"\nClass distribution in training set:")
print(labels_train['x'].value_counts().sort_index())
print(f"\nClass distribution in test set:")
print(labels_test['x'].value_counts().sort_index())

# Save all the split data
np.save(f'C_train.npy', C_train)
np.save(f'C_test.npy', C_test)
np.save('train_indices.npy', train_indices)
np.save('test_indices.npy', test_indices)

# Save labels as CSV files
labels_train.to_csv('labels_train.csv', index=False)
labels_test.to_csv('labels_test.csv', index=False)

Original C_reduced shape: (10000, 3000)
Original labels shape: (10000, 1)
Total number of samples: 10000
Training samples: 9000
Test samples: 1000

Train data shape: (9000, 3000)
Test data shape: (1000, 3000)
Train labels shape: (9000, 1)
Test labels shape: (1000, 1)

Class distribution in training set:
x
ASDC                    5
B intermediate        136
B memory              187
B naive               437
CD4 CTL                99
CD4 Naive            1019
CD4 Proliferating       6
CD4 TCM               864
CD4 TEM               248
CD8 Naive             623
CD8 Proliferating       5
CD8 TCM               166
CD8 TEM               649
CD14 Mono            2435
CD16 Mono             359
Eryth                   4
HSPC                   19
ILC                     5
MAIT                  153
NK                    841
NK Proliferating       30
NK_CD56bright          39
Plasmablast            20
Platelet              110
Treg                  146
cDC1                    9
cDC2             